## Dataset Creation

In [5]:
import numpy as np
import pandas as pd


# config
RANDOM_SEED = 42
N_SAMPLES = 5000
N_FEATURES = 100
N_RELEVANT_FEATURES = 10
NOISE_STD = 1.0

np.random.seed(RANDOM_SEED)

# all features are sampled from a standard Gaussian distribution
X = np.random.normal(loc=0, scale=1, size=(N_SAMPLES, N_FEATURES))

# define true coefficients
## first N_RELEVANT_FEATURES features are relevant
## remaining features have true coefficient 0
true_coefficients = np.zeros(N_FEATURES)
true_coefficients[:N_RELEVANT_FEATURES] = np.array([
    5.0, -4.0, 3.5, -3.0, 2.5,
    -2.0, 1.5, -1.25, 1.0, -0.75
])

# generate labels
noise = np.random.normal(loc=0, scale=NOISE_STD, size=N_SAMPLES)
y = X @ true_coefficients + noise

# create df
feature_names = [f"x{i+1}" for i in range(N_FEATURES)]

df = pd.DataFrame(X, columns=feature_names)
df["y"] = y

# save ds
df.to_csv("control_ds.csv", index=False)

# save the true coefficients
coef_df = pd.DataFrame({
    "feature": feature_names,
    "true_coefficient": true_coefficients,
    "is_relevant": true_coefficients != 0
})

coef_df.to_csv("true_coefficients.csv", index=False)

## Experiment

In [6]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score

# load DS
df = pd.read_csv("control_ds.csv")
coef_df = pd.read_csv("true_coefficients.csv")
feature_names = [col for col in df.columns if col != "y"]

# get data
X = df[feature_names].values
y = df["y"].values
true_coefficients = coef_df["true_coefficient"].values
true_relevant = coef_df["is_relevant"].values.astype(bool)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# model config
ALPHA = 0.1
lasso_model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=ALPHA, max_iter=10000, random_state=42)
)
ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=ALPHA)
)

# train & get coefficients
lasso_model.fit(X_train, y_train)
ridge_model.fit(X_train, y_train)
lasso_coefficients = lasso_model.named_steps["lasso"].coef_
ridge_coefficients = ridge_model.named_steps["ridge"].coef_

# predictions
lasso_preds = lasso_model.predict(X_test)
ridge_preds = ridge_model.predict(X_test)


# eval helper
def evaluate_model(name, coefficients, predictions):
    exact_zero = coefficients == 0

    predicted_relevant = coefficients != 0
    predicted_irrelevant = coefficients == 0

    true_positive = np.sum(predicted_relevant & true_relevant)
    false_positive = np.sum(predicted_relevant & ~true_relevant)
    true_negative = np.sum(predicted_irrelevant & ~true_relevant)
    false_negative = np.sum(predicted_irrelevant & true_relevant)

    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    print(f"\n{name}")
    print("-" * 10)
    print(f"Test MSE: {mse:.4f}")
    print(f"Test R^2:  {r2:.4f}")
    print(f"Number of exact-zero coefficients: {np.sum(exact_zero)} / {len(coefficients)}")
    print(f"Number of nonzero coefficients:    {np.sum(~exact_zero)} / {len(coefficients)}")

    print("\nFeature-selection comparison:")
    print(f"True positives:  {true_positive}  relevant features kept")
    print(f"False positives: {false_positive}  irrelevant features kept")
    print(f"True negatives:  {true_negative}  irrelevant features zeroed")
    print(f"False negatives: {false_negative}  relevant features zeroed")

    return {
        "model": name,
        "mse": mse,
        "r2": r2,
        "exact_zero_coefficients": np.sum(exact_zero),
        "nonzero_coefficients": np.sum(~exact_zero),
        "true_positives": true_positive,
        "false_positives": false_positive,
        "true_negatives": true_negative,
        "false_negatives": false_negative
    }


# eval models
lasso_results = evaluate_model("Lasso", lasso_coefficients, lasso_preds)
ridge_results = evaluate_model("Ridge", ridge_coefficients, ridge_preds)


# save
results_df = pd.DataFrame({
    "feature": feature_names,
    "true_coefficient": true_coefficients,
    "is_relevant": true_relevant,
    "lasso_coefficient": lasso_coefficients,
    "ridge_coefficient": ridge_coefficients,
    "lasso_exact_zero": lasso_coefficients == 0,
    "ridge_exact_zero": ridge_coefficients == 0
})

results_df.to_csv("results.csv", index=False)

print("\nSaved coefficient results to coefficient_results.csv")


Lasso
----------
Test MSE: 1.0917
Test R^2:  0.9864
Number of exact-zero coefficients: 90 / 100
Number of nonzero coefficients:    10 / 100

Feature-selection comparison:
True positives:  10  relevant features kept
False positives: 0  irrelevant features kept
True negatives:  90  irrelevant features zeroed
False negatives: 0  relevant features zeroed

Ridge
----------
Test MSE: 1.0334
Test R^2:  0.9872
Number of exact-zero coefficients: 0 / 100
Number of nonzero coefficients:    100 / 100

Feature-selection comparison:
True positives:  10  relevant features kept
False positives: 90  irrelevant features kept
True negatives:  0  irrelevant features zeroed
False negatives: 0  relevant features zeroed

Saved coefficient results to coefficient_results.csv


#### Explanation:

True positive: A truly relevant feature was kept by the model
False positive: An irrelevant feature was kept by the model

True negative: An irrelevant feature was correctly zeroed out
False negative: A truly relevant feature was incorrectly zeroed out